# Partie D : système SAV multi-agents
## 13a : agent_garantie et outil local consulter_garantie

In [1]:
import calendar
import json
from datetime import date, datetime
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv(override=True)


def trouver_racine() -> Path:
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / "pyproject.toml").exists():
            return d
    raise FileNotFoundError("pyproject.toml introuvable")


RACINE = trouver_racine()
REGISTRE = RACINE / "sav" / "registre_garanties.json"


def ajouter_mois(d: date, n: int) -> date:
    mois = d.month - 1 + n
    annee, mois = d.year + mois // 12, mois % 12 + 1
    jour = min(d.day, calendar.monthrange(annee, mois)[1])
    return date(annee, mois, jour)


@tool
def consulter_garantie(numero_serie: str) -> dict:
    """Consulte le registre des garanties à partir d'un numéro de série (ex: PX200-0042).
    Renvoie le produit, la date d'achat, la date de fin et si la garantie est active."""
    with open(REGISTRE, encoding="utf-8") as f:  # lecture seule
        registre = json.load(f)
    for ligne in registre:
        if ligne["numero_serie"].upper() == numero_serie.strip().upper():
            achat = datetime.strptime(ligne["date_achat"], "%Y-%m-%d").date()
            fin = ajouter_mois(achat, ligne["duree_mois"])
            return {
                "numero_serie": ligne["numero_serie"],
                "produit": ligne["produit"],
                "date_achat": str(achat),
                "date_fin_garantie": str(fin),
                "sous_garantie": date.today() <= fin,
                "date_du_jour": str(date.today()),
            }
    return {"erreur": f"Numéro de série {numero_serie} introuvable dans le registre."}


print(RACINE)
print(REGISTRE.exists())

C:\Users\bouho\Desktop\tp5_agents
True


In [2]:
print(consulter_garantie.invoke({"numero_serie": "PX200-0042"}))
print(consulter_garantie.invoke({"numero_serie": "PX200-0007"}))
print(consulter_garantie.invoke({"numero_serie": "XXX"}))

{'numero_serie': 'PX200-0042', 'produit': 'Imprimante PX-200', 'date_achat': '2025-06-15', 'date_fin_garantie': '2027-06-15', 'sous_garantie': True, 'date_du_jour': '2026-11-28'}
{'numero_serie': 'PX200-0007', 'produit': 'Imprimante PX-200', 'date_achat': '2023-03-10', 'date_fin_garantie': '2025-03-10', 'sous_garantie': False, 'date_du_jour': '2026-11-28'}
{'erreur': 'Numéro de série XXX introuvable dans le registre.'}


In [3]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

agent_garantie = create_agent(
    model=llm,
    tools=[consulter_garantie],
    system_prompt=(
        "Tu es l'agent garantie du service après-vente d'un distributeur d'électronique. "
        "Pour toute question de garantie, appelle l'outil consulter_garantie avec le "
        "numéro de série donné par le client. Réponds en français, de façon courte : "
        "indique si le produit est sous garantie et la date de fin. "
        "N'invente jamais de données : si l'outil renvoie une erreur, dis-le au client."
    ),
    name="agent_garantie",
)

resp = agent_garantie.invoke(
    {"messages": [{"role": "user", "content":
        "Est-ce que mon imprimante est encore sous garantie ? Son numéro de série est PX200-0042."}]}
)
for m in resp["messages"]:
    m.pretty_print()

================================ Human Message =================================

Est-ce que mon imprimante est encore sous garantie ? Son numéro de série est PX200-0042.
================================== Ai Message ==================================
Name: agent_garantie
Tool Calls:
  consulter_garantie (965eb944-3194-4801-b564-b9edd97267f2)
 Call ID: 965eb944-3194-4801-b564-b9edd97267f2
  Args:
    numero_serie: PX200-0042
================================= Tool Message =================================
Name: consulter_garantie

{"numero_serie": "PX200-0042", "produit": "Imprimante PX-200", "date_achat": "2025-06-15", "date_fin_garantie": "2027-06-15", "sous_garantie": true, "date_du_jour": "2026-11-28"}
================================== Ai Message ==================================
Name: agent_garantie

Oui, votre imprimante est encore sous garantie jusqu'au 15 juin 2027.
